# Rerun of Top 10 Configurations (adult/pen-based)

## Imports

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from pathlib import Path

from Parser import Parser
from IBL import IBL
from processing_types import (
    NormalizationStrategy, EncodingStrategy,
    MissingValuesNumericStrategy, MissingValuesCategoricalStrategy, RetentionPolicy
)

BASE = "../datasetsCBR/datasetsCBR"
NUM_SPLITS = 10
ENCODING_FOR_METRIC ={
        "euclidean": EncodingStrategy.ONE_HOT_ENCODE,
        "cosine":    EncodingStrategy.ONE_HOT_ENCODE,
        "heom": EncodingStrategy.LABEL_ENCODE, 
}

DIR = Path("./results")
ADULT_PATH = DIR / "shortlist_for_rerun_adult.csv"
PENBASED_PATH = DIR / "shortlist_for_rerun_pen-based.csv"

## Adult

In [ ]:

def run_suite(
    dataset_name: str,
    k: int,
    metric: str,
    run_retentions: str,
    votes: str
):
    enc_strategy = ENCODING_FOR_METRIC[metric]

    parser = Parser(
        base_path=BASE,
        dataset_name=dataset_name,
        normalization_strategy=NormalizationStrategy.MEAN_NORMALIZE,
        encoding_strategy=enc_strategy, 
        missing_values_numeric_strategy=MissingValuesNumericStrategy.MEDIAN,
        missing_values_categorical_strategy=MissingValuesCategoricalStrategy.MODE,
        num_splits=NUM_SPLITS,
    )
    types = parser.get_types()

    splits = [parser.get_split(fold) for fold in range(NUM_SPLITS)]
    
    for fold, (train_matrix, test_matrix) in enumerate(splits):
        # print(f"metric={metric} | k={k} | vote={vote} | retention={retention} | fold={fold}")

        ibl = IBL()

        # Fit + predict
        t0 = time.perf_counter()
        ibl.fit(train_matrix)
        t1 = time.perf_counter()
        preds = ibl.run(
            test_matrix,
            k=k,
            metric=metric,
            vote=vote,
            retention_policy=retention,
            types=types
        )
        t2 = time.perf_counter()

        # Times
        fit_times.append(t1 - t0)
        pred_times.append(t2 - t1)
        total_times.append(t2 - t0)

        # Metrics

        y_true = test_matrix.iloc[:, -1].to_numpy()
        y_pred = np.asarray(preds)

        accs.appe



## Adult Rerun

In [ ]:
df_cfg = pd.read_csv(ADULT_PATH)

configs = [
    {
        "dataset":   row["dataset"],
        "metric":    row["metric"],     # distance metric
        "k":         int(row["k"]),
        "vote":      row["vote"],
        "retention": row["retention"],
    }
    for _, row in df_cfg.iterrows()
]

RET_MAP  = {
    "RetentionPolicy.ALWAYS_RETAIN": RetentionPolicy.ALWAYS_RETAIN,
    "RetentionPolicy.NEVER_RETAIN": RetentionPolicy.NEVER_RETAIN,
    "RetentionPolicy.DIFFERENT_CLASS_RETENTION": RetentionPolicy.DIFFERENT_CLASS_RETENTION,
    "RetentionPolicy.DD_RETENTION": RetentionPolicy.DD_RETENTION,
}

for cfg in configs:
    print(f"dataset={cfg['dataset']}, metric={cfg['metric']}, k={cfg['k']}, vote={cfg['vote']}, retention={cfg['retention']}")
    run_suite(cfg['dataset'], k=cfg['k'], metric=cfg['metric'], run_retentions=RET_MAP[cfg['retention']], votes=cfg['vote'])

dataset=adult, metric=cosine, k=7, vote=borda, retention=RetentionPolicy.ALWAYS_RETAIN
dataset=adult, metric=cosine, k=7, vote=borda, retention=RetentionPolicy.DD_RETENTION
dataset=adult, metric=cosine, k=7, vote=borda, retention=RetentionPolicy.DIFFERENT_CLASS_RETENTION
dataset=adult, metric=cosine, k=5, vote=borda, retention=RetentionPolicy.ALWAYS_RETAIN
dataset=adult, metric=cosine, k=5, vote=borda, retention=RetentionPolicy.DD_RETENTION
dataset=adult, metric=cosine, k=5, vote=modified_plurality, retention=RetentionPolicy.ALWAYS_RETAIN
dataset=adult, metric=cosine, k=7, vote=borda, retention=RetentionPolicy.NEVER_RETAIN
dataset=adult, metric=cosine, k=5, vote=modified_plurality, retention=RetentionPolicy.DD_RETENTION
dataset=adult, metric=cosine, k=5, vote=borda, retention=RetentionPolicy.DIFFERENT_CLASS_RETENTION
dataset=adult, metric=cosine, k=5, vote=modified_plurality, retention=RetentionPolicy.DIFFERENT_CLASS_RETENTION
